<a href="https://colab.research.google.com/github/Tinopino/AML_Pipeline/blob/main/AML_insurance_preprocessing_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-to-end preprocessing pipeline for Dutch insurance policy PDFs

This script is written in a notebook-style format for Google Colab/Jupyter.
It creates reproducible datasets for Dutch insurance policy condition PDFs:

1. `documents` table (1 row per PDF)
2. `pages` table (1 row per page)
3. `chunks` table (many rows per PDF) with traceability metadata
4. Optional embeddings + clustering artifacts for taxonomy bootstrapping

**Important constraints implemented:**
- Uses PyMuPDF (`fitz`) for extraction with block-level coordinates.
- No OCR, no LLM.
- Dutch-aware lexical preprocessing via spaCy `nl_core_news_sm`.
- Keeps legal punctuation/numbers in `text_raw` for embeddings.

## 1) Install/Import dependencies

In Colab, uncomment the `pip install` lines if needed.
The script handles optional dependencies gracefully.

In [ ]:
!pip -q install pymupdf pandas tqdm spacy sentence-transformers scikit-learn matplotlib
!python -m spacy download nl_core_news_sm
!pip -q install ydata-profiling wordcloud  # optional

import os
import re
import json
import math
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
import fitz  # PyMuPDF

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

# Optional libs
try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except ImportError:
    SENTENCE_TRANSFORMERS_AVAILABLE = False
    warnings.warn("sentence-transformers not installed. Embeddings/clustering will be skipped.")

try:
    import spacy
    SPACY_AVAILABLE = True
except ImportError:
    SPACY_AVAILABLE = False
    warnings.warn("spaCy not installed. text_lexical generation will be skipped.")

try:
    from ydata_profiling import ProfileReport
    YDATA_AVAILABLE = True
except ImportError:
    YDATA_AVAILABLE = False

try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    WORDCLOUD_AVAILABLE = False

try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except ImportError:
    HDBSCAN_AVAILABLE = False

Extract the car insurance files from the bigger dataset:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_dir = Path("/content/drive/MyDrive/Applied ML/AML_Pipeline_files")

excel_path = base_dir / "polisvoorwaarden_aantallen_auto.xlsx"
zip_path   = base_dir / "voorbeelden_polisvoorwaarden.zip"
out_dir    = base_dir / "car_pdfs"

out_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(excel_path)

# Make Excel filenames uppercase
target_pdfs = (
    df["document"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .tolist()
)

target_set = set(target_pdfs)

found = []
missing = []

with zipfile.ZipFile(zip_path, "r") as zf:
    zip_members = [m for m in zf.namelist() if not m.endswith("/")]
    basename_map = {}

    for member in zip_members:
        # Also make zip basenames uppercase
        basename = Path(member).name.upper()
        basename_map.setdefault(basename, []).append(member)

    for pdf_name in sorted(target_set):
        matches = basename_map.get(pdf_name, [])

        if len(matches) == 0:
            missing.append(pdf_name)
            continue

        chosen_member = sorted(matches, key=lambda x: (x.count("/"), len(x)))[0]

        # Save extracted file with uppercase filename
        p = Path(pdf_name)
        output_name = p.stem.upper() + p.suffix.lower()   # stem upper, .pdf stays lowercase
        output_path = out_dir / output_name

        with zf.open(chosen_member) as src, open(output_path, "wb") as dst:
            shutil.copyfileobj(src, dst)

        found.append(pdf_name)

print(f"Done. Extracted {len(found)} PDFs to: {out_dir}")
print(f"Missing: {len(missing)}")

In [ ]:


def ensure_pdf_dir(config: Dict[str, Any]) -> Path:
    """
    If PDF_DIR points to a .zip file, unzip it and return the extracted folder path.
    Otherwise, return PDF_DIR as-is.
    """
    pdf_path = Path(config["PDF_DIR"])

    if pdf_path.suffix.lower() == ".zip":
        extract_dir = pdf_path.with_suffix("")  # remove .zip
        if not extract_dir.exists():
            print(f"Unzipping {pdf_path} -> {extract_dir}")
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(pdf_path, "r") as zf:
                zf.extractall(extract_dir)
        else:
            print(f"ZIP already extracted at {extract_dir}")

        config["PDF_DIR"] = str(extract_dir.resolve())
        return extract_dir

    return pdf_path
PDF_DIR = ensure_pdf_dir(CONFIG)
pdf_paths = get_pdf_paths(PDF_DIR)
print(f"Found {len(pdf_paths)} PDF files in {PDF_DIR}")

## 2) Configuration

Set `PDF_DIR` and `OUT_DIR` before running extraction.

Modes:
- **A) Local/Drive folder input**: set `PDF_DIR` directly.
- **B) Colab upload fallback**: optional helper cell below.

In [ ]:
CONFIG: Dict[str, Any] = {
    "PDF_DIR": "/content/drive/MyDrive/Applied ML/AML_Pipeline_files/car_pdfs",  # REQUIRED: path to folder with PDFs
    "OUT_DIR": "./outputs_dutch_policy_pipeline",  # REQUIRED: output folder
    "USE_COLAB_UPLOAD_FALLBACK": False,
    "SAVE_FULL_PAGE_TEXT": False,
    "DOC_MIN_TOTAL_CHARS": 2000,
    "PAGE_LOW_TEXT_CHARS": 200,
    "MAX_LOW_TEXT_PAGE_FRAC": 0.5,
    "REPEATED_LINE_PAGE_FRAC": 0.5,
    "REPEATED_LINE_MAX_CHARS": 80,
    "BLOCK_MERGE_VERTICAL_GAP": 12.0,
    "BLOCK_MERGE_X_TOL": 20.0,
    "MAX_CHARS_FOR_SPACY": 2000,
    "SPACY_BATCH_SIZE": 64,
    "EMBED_MODEL": "paraphrase-multilingual-MiniLM-L12-v2",
    "EMBED_BATCH_SIZE": 64,
    "K_VALUES": [10, 20, 40],
    "PCA_SCATTER_K": 20,
    "HDBSCAN_MIN_CLUSTER_SIZE": 30,
    "WORDCLOUD_TOP_CLUSTERS": 5,
    "WORDCLOUD_MAX_WORDS": 120,
    "RANDOM_SEED": 42,
}
CONFIG.update({
    "CHUNK_MODE": "overlap",
    "CHUNK_TARGET_CHARS": 1200,
    "CHUNK_MIN_CHARS": 400,
    "CHUNK_OVERLAP_CHARS": 250,
    "ALLOW_CROSS_PAGE_CHUNKS": False,
})

np.random.seed(CONFIG["RANDOM_SEED"])

OUT_DIR = Path(CONFIG["OUT_DIR"])
(OUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "reports").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "wordclouds").mkdir(parents=True, exist_ok=True)

## 3) Optional Colab upload fallback (Mode B)

If `PDF_DIR` is not set and you're in Colab, you can upload PDFs.

In [ ]:
def optional_colab_upload_fallback(config: Dict[str, Any]) -> None:
    """Optional uploader for Colab if PDF_DIR is not set."""
    if config.get("PDF_DIR"):
        return
    if not config.get("USE_COLAB_UPLOAD_FALLBACK", False):
        return

    try:
        from google.colab import files
    except ImportError:
        warnings.warn("Colab upload fallback enabled, but not running in Colab.")
        return

    upload_dir = Path("./uploaded_pdfs")
    upload_dir.mkdir(parents=True, exist_ok=True)

    print("Upload one or more PDF files...")
    uploaded = files.upload()
    for fname, content in uploaded.items():
        if fname.lower().endswith(".pdf"):
            with open(upload_dir / fname, "wb") as f:
                f.write(content)
    config["PDF_DIR"] = str(upload_dir.resolve())
    print(f"PDF_DIR set to: {config['PDF_DIR']}")


optional_colab_upload_fallback(CONFIG)

if not CONFIG["PDF_DIR"]:
    raise ValueError("Please set CONFIG['PDF_DIR'] to a folder containing PDFs.")

PDF_DIR = Path(CONFIG["PDF_DIR"])
if not PDF_DIR.exists() or not PDF_DIR.is_dir():
    raise FileNotFoundError(f"PDF_DIR does not exist or is not a directory: {PDF_DIR}")

## 4) Helper functions

Core utilities for cleaning, block handling, heading/list detection,
lexical preprocessing, and reporting.

In [ ]:
def light_clean_text(text: str) -> str:
    """Light, legal-safe cleaning: soft hyphen removal, hyphenated linebreak repair, whitespace normalize."""
    if not text:
        return ""
    t = text.replace("\u00ad", "")
    t = re.sub(r"-\s*\n\s*", "", t)
    t = t.replace("\r", "\n")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


def split_lines_clean(text: str) -> List[str]:
    """Split into non-empty stripped lines."""
    return [ln.strip() for ln in text.splitlines() if ln.strip()]


def detect_repeated_header_footer_lines(
    page_texts: List[str],
    min_page_frac: float = 0.5,
    max_chars: int = 80,
) -> List[str]:
    """Detect repeated short lines likely to be header/footer lines within a document."""
    n_pages = len(page_texts)
    if n_pages == 0:
        return []

    line_docfreq: Dict[str, int] = {}
    for text in page_texts:
        unique_lines = set(split_lines_clean(text))
        for ln in unique_lines:
            if len(ln) <= max_chars:
                line_docfreq[ln] = line_docfreq.get(ln, 0) + 1

    threshold = max(2, math.ceil(min_page_frac * n_pages))
    repeated = [ln for ln, df in line_docfreq.items() if df >= threshold]
    return sorted(repeated)


def remove_lines(text: str, remove_set: set) -> str:
    """Remove exact-match lines from page text."""
    out_lines = [ln for ln in text.splitlines() if ln.strip() not in remove_set]
    return "\n".join(out_lines)


def get_pdf_paths(pdf_dir: Path) -> List[Path]:
    """List PDF files in a folder."""
    return sorted([p for p in pdf_dir.iterdir() if p.is_file() and p.suffix.lower() == ".pdf"])


def is_heading(text: str) -> bool:
    """Heuristic heading detection."""
    t = text.strip()
    if not t:
        return False
    if len(t) <= 90 and not t.endswith("."):
        return True
    if re.match(r"^(artikel|hoofdstuk|paragraaf)\b", t.lower()):
        return True
    if re.match(r"^\d+(\.\d+){0,3}\s+", t):
        return True
    if re.match(r"^\d+(\.\d+){0,3}$", t):
        return True
    return False


def is_list_item(text: str) -> bool:
    """Heuristic list item detection."""
    t = text.strip()
    if not t:
        return False
    patterns = [
        r"^[-•‣▪]\s+",
        r"^\d+[\.)]\s+",
        r"^[a-zA-Z][\.)]\s+",
        r"^\([0-9a-zA-Z]+\)\s+",
    ]
    return any(re.match(p, t) for p in patterns)


def bbox_union(bboxes: List[Tuple[float, float, float, float]]) -> Tuple[float, float, float, float]:
    """Union of multiple bounding boxes."""
    x0 = min(b[0] for b in bboxes)
    y0 = min(b[1] for b in bboxes)
    x1 = max(b[2] for b in bboxes)
    y1 = max(b[3] for b in bboxes)
    return (x0, y0, x1, y1)


def blocks_reading_order(blocks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Sort blocks top-to-bottom then left-to-right."""
    return sorted(blocks, key=lambda b: (round(b["y0"], 1), round(b["x0"], 1)))


def merge_blocks(blocks: List[Dict[str, Any]], vertical_gap: float, x_tol: float) -> List[List[Dict[str, Any]]]:
    """Merge adjacent blocks likely belonging to same paragraph."""
    merged: List[List[Dict[str, Any]]] = []
    if not blocks:
        return merged

    current = [blocks[0]]
    for b in blocks[1:]:
        prev = current[-1]
        v_gap = b["y0"] - prev["y1"]
        x_close = abs(b["x0"] - prev["x0"]) <= x_tol
        if v_gap <= vertical_gap and x_close and not is_heading(prev["text"]) and not is_heading(b["text"]):
            current.append(b)
        else:
            merged.append(current)
            current = [b]
    merged.append(current)
    return merged


def chunk_type_for_text(text: str) -> str:
    """Assign chunk type from text heuristics."""
    if is_heading(text):
        return "heading"
    if is_list_item(text):
        return "list_item"
    if len(text.strip()) == 0:
        return "other"
    return "paragraph"


def make_section_path(section_stack: List[str]) -> str:
    """Serialize section stack."""
    return " > ".join(section_stack)


def safe_filename(s: str) -> str:
    """Safe filename helper."""
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", s)

## 5) spaCy Dutch pipeline for lexical field (`text_lexical`)

Used only for lexical analytics (TF-IDF/word clouds), not for embedding text.

In [ ]:
NLP = None
if SPACY_AVAILABLE:
    try:
        NLP = spacy.load("nl_core_news_sm", disable=["parser", "ner", "textcat"])
    except OSError:
        warnings.warn("spaCy Dutch model 'nl_core_news_sm' is not installed. Install via: python -m spacy download nl_core_news_sm")
        NLP = None


def build_text_lexical(text: str, max_chars: int = 2000) -> str:
    """Build Dutch lexical string: lemmatized + stopwords/punctuation removed."""
    if NLP is None:
        return ""
    if not text:
        return ""
    t = text
    if len(t) > max_chars:
        warnings.warn("Chunk too long for spaCy lexical processing; truncating.")
        t = t[:max_chars]

    doc = NLP(t)
    lemmas = []
    for token in doc:
        if token.is_space or token.is_punct or token.is_stop:
            continue
        lemma = token.lemma_.strip().lower()
        if lemma:
            lemmas.append(lemma)
    return " ".join(lemmas)

## 6) PDF extraction + per-document header/footer detection

Produces:
- `documents_df`
- `pages_df`
- intermediate block records for chunking

In [ ]:
@dataclass
class BlockRecord:
    doc_id: str
    filename: str
    page_no: int
    block_id: str
    x0: float
    y0: float
    x1: float
    y1: float
    text_raw_block: str


def extract_pdf(doc_path: Path, config: Dict[str, Any]) -> Tuple[Dict[str, Any], List[Dict[str, Any]], List[BlockRecord], List[str]]:
    """Extract document/page/block data from one PDF."""
    doc_id = doc_path.stem
    notes = []
    pages_records: List[Dict[str, Any]] = []
    block_records: List[BlockRecord] = []
    page_texts_for_repetition: List[str] = []

    try:
        pdf = fitz.open(doc_path)
        page_count = len(pdf)
        total_chars = 0

        for pidx in range(page_count):
            page = pdf[pidx]
            page_no = pidx + 1
            page_text = page.get_text("text") or ""
            page_text_clean = light_clean_text(page_text)
            page_texts_for_repetition.append(page_text_clean)
            page_char_count = len(page_text_clean)
            total_chars += page_char_count

            raw_blocks = page.get_text("blocks")
            for bidx, b in enumerate(raw_blocks):
                x0, y0, x1, y1, btext, *_ = b
                btext_clean = light_clean_text(btext or "")
                if not btext_clean:
                    continue
                block_records.append(
                    BlockRecord(
                        doc_id=doc_id,
                        filename=doc_path.name,
                        page_no=page_no,
                        block_id=f"{doc_id}_p{page_no}_b{bidx}",
                        x0=float(x0),
                        y0=float(y0),
                        x1=float(x1),
                        y1=float(y1),
                        text_raw_block=btext_clean,
                    )
                )

            pages_records.append(
                {
                    "doc_id": doc_id,
                    "filename": doc_path.name,
                    "page_no": page_no,
                    "page_char_count": page_char_count,
                    "page_text_raw": page_text_clean,
                }
            )

        low_text_pages = sum(1 for r in pages_records if r["page_char_count"] < config["PAGE_LOW_TEXT_CHARS"])
        low_text_frac = (low_text_pages / page_count) if page_count else 1.0

        extraction_ok = True
        if total_chars < config["DOC_MIN_TOTAL_CHARS"]:
            extraction_ok = False
            notes.append(f"total_chars<{config['DOC_MIN_TOTAL_CHARS']}")
        if low_text_frac > config["MAX_LOW_TEXT_PAGE_FRAC"]:
            extraction_ok = False
            notes.append(f"low_text_page_frac>{config['MAX_LOW_TEXT_PAGE_FRAC']}")

        doc_record = {
            "doc_id": doc_id,
            "filename": doc_path.name,
            "page_count": page_count,
            "char_count_total": total_chars,
            "extraction_ok": extraction_ok,
            "notes": ";".join(notes),
        }

        return doc_record, pages_records, block_records, page_texts_for_repetition

    except Exception as e:
        return {
            "doc_id": doc_id,
            "filename": doc_path.name,
            "page_count": 0,
            "char_count_total": 0,
            "extraction_ok": False,
            "notes": f"error:{repr(e)}",
        }, [], [], []


pdf_paths = get_pdf_paths(PDF_DIR)
if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in: {PDF_DIR}")

all_doc_records: List[Dict[str, Any]] = []
all_page_records: List[Dict[str, Any]] = []
all_block_records: List[BlockRecord] = []
header_footer_map: Dict[str, Dict[str, Any]] = {}

for pdf_path in tqdm(pdf_paths, desc="Extracting PDFs"):
    doc_record, page_records, block_records, page_texts = extract_pdf(pdf_path, CONFIG)
    all_doc_records.append(doc_record)
    all_page_records.extend(page_records)
    all_block_records.extend(block_records)

    repeated = detect_repeated_header_footer_lines(
        page_texts,
        min_page_frac=CONFIG["REPEATED_LINE_PAGE_FRAC"],
        max_chars=CONFIG["REPEATED_LINE_MAX_CHARS"],
    )
    header_footer_map[doc_record["doc_id"]] = {
        "doc_id": doc_record["doc_id"],
        "filename": doc_record["filename"],
        "repeated_lines": repeated,
    }

# Attach suspected header/footer lines per page and create cleaned page text
for p in all_page_records:
    doc_id = p["doc_id"]
    repeated = set(header_footer_map.get(doc_id, {}).get("repeated_lines", []))
    lines = split_lines_clean(p.get("page_text_raw", ""))
    suspected = [ln for ln in lines if ln in repeated]
    p["suspected_header_footer_lines"] = suspected
    p["page_text_no_hf"] = light_clean_text(remove_lines(p.get("page_text_raw", ""), repeated))


documents_df = pd.DataFrame(all_doc_records)
pages_df = pd.DataFrame(all_page_records)
blocks_df = pd.DataFrame([b.__dict__ for b in all_block_records])

print(documents_df.head(3))
print(pages_df.head(3))
print(blocks_df.head(3))

## 7) Layout-aware chunking with traceability

Chunk units are created from merged blocks and labeled as heading/list_item/paragraph/other.
A best-effort `section_path` is maintained via a running heading stack.

In [ ]:
def build_atomic_units(
    doc_id: str,
    pages_df_doc: pd.DataFrame,
    blocks_df_doc: pd.DataFrame,
    repeated: set,
    config: Dict[str, Any],
) -> List[Dict[str, Any]]:
    units = []
    section_stack: List[str] = []

    for page_no in sorted(pages_df_doc["page_no"].unique().tolist()):
        page_blocks = blocks_df_doc[blocks_df_doc["page_no"] == page_no].copy()
        if page_blocks.empty:
            continue

        page_blocks["text_clean_no_hf"] = page_blocks["text_raw_block"].apply(
            lambda t: light_clean_text(remove_lines(t, repeated))
        )
        page_blocks = page_blocks[page_blocks["text_clean_no_hf"].str.len() > 0]
        if page_blocks.empty:
            continue

        block_items = [
            {
                "block_id": row.block_id,
                "x0": float(row.x0),
                "y0": float(row.y0),
                "x1": float(row.x1),
                "y1": float(row.y1),
                "text": row.text_clean_no_hf,
            }
            for row in page_blocks.itertuples(index=False)
        ]

        block_items = blocks_reading_order(block_items)
        merged_groups = merge_blocks(
            block_items,
            vertical_gap=config["BLOCK_MERGE_VERTICAL_GAP"],
            x_tol=config["BLOCK_MERGE_X_TOL"],
        )

        for group in merged_groups:
            text = light_clean_text("\n".join(x["text"] for x in group))
            if not text:
                continue

            ctype = chunk_type_for_text(text)

            if ctype == "heading":
                heading = text.strip()
                m = re.match(r"^(\d+(?:\.\d+)*)", heading)
                if m:
                    depth = len(m.group(1).split("."))
                    section_stack = section_stack[: max(0, depth - 1)]
                    section_stack.append(heading)
                else:
                    if len(section_stack) >= 6:
                        section_stack = section_stack[-5:]
                    section_stack.append(heading)

            bboxes = [(x["x0"], x["y0"], x["x1"], x["y1"]) for x in group]
            ux0, uy0, ux1, uy1 = bbox_union(bboxes)

            units.append({
                "doc_id": doc_id,
                "page_no": page_no,
                "block_ids": [x["block_id"] for x in group],
                "bbox": [ux0, uy0, ux1, uy1],
                "text": text,
                "chunk_type_layout": ctype,
                "section_path": make_section_path(section_stack),
            })

    return units

In [ ]:
def make_overlapping_chunks(
    doc_id: str,
    units: List[Dict[str, Any]],
    config: Dict[str, Any],
) -> List[Dict[str, Any]]:
    chunks = []
    order_index = 0

    target_chars = config["CHUNK_TARGET_CHARS"]
    min_chars = config["CHUNK_MIN_CHARS"]
    overlap_chars = config["CHUNK_OVERLAP_CHARS"]
    allow_cross_page = config["ALLOW_CROSS_PAGE_CHUNKS"]

    start = 0
    n = len(units)

    while start < n:
        current_units = []
        total_len = 0
        start_page = units[start]["page_no"]

        i = start
        while i < n:
            unit = units[i]

            if not allow_cross_page and current_units and unit["page_no"] != start_page:
                break

            unit_len = len(unit["text"])
            if current_units and total_len >= target_chars:
                break

            current_units.append(unit)
            total_len += unit_len + 2
            i += 1

        if not current_units:
            start += 1
            continue

        chunk_text = "\n\n".join(u["text"] for u in current_units).strip()
        if len(chunk_text) < min_chars and i < n:
            while i < n and len(chunk_text) < min_chars:
                unit = units[i]
                if not allow_cross_page and unit["page_no"] != start_page:
                    break
                current_units.append(unit)
                chunk_text = "\n\n".join(u["text"] for u in current_units).strip()
                i += 1

        all_bboxes = [tuple(u["bbox"]) for u in current_units]
        ux0, uy0, ux1, uy1 = bbox_union(all_bboxes)

        chunks.append({
            "doc_id": doc_id,
            "chunk_id": f"{doc_id}_c{order_index:06d}",
            "page_start": current_units[0]["page_no"],
            "page_end": current_units[-1]["page_no"],
            "block_ids": [bid for u in current_units for bid in u["block_ids"]],
            "bbox": [ux0, uy0, ux1, uy1],
            "section_path": current_units[-1]["section_path"],
            "chunk_type_layout": "mixed" if len(set(u["chunk_type_layout"] for u in current_units)) > 1 else current_units[0]["chunk_type_layout"],
            "order_index": order_index,
            "text_raw": chunk_text,
        })
        order_index += 1

        # slide start forward so we keep overlap_chars worth of text
        back_chars = 0
        new_start = len(current_units) - 1
        for j in range(len(current_units) - 1, -1, -1):
            back_chars += len(current_units[j]["text"]) + 2
            new_start = j
            if back_chars >= overlap_chars:
                break

        start = max(start + 1, start + new_start)

    return chunks

In [ ]:
def chunk_document(
    doc_id: str,
    pages_df_doc: pd.DataFrame,
    blocks_df_doc: pd.DataFrame,
    config: Dict[str, Any],
) -> List[Dict[str, Any]]:
    repeated = set(header_footer_map.get(doc_id, {}).get("repeated_lines", []))

    units = build_atomic_units(
        doc_id=doc_id,
        pages_df_doc=pages_df_doc,
        blocks_df_doc=blocks_df_doc,
        repeated=repeated,
        config=config,
    )

    return make_overlapping_chunks(
        doc_id=doc_id,
        units=units,
        config=config,
    )

## 8) Build lexical text field (`text_lexical`) for analytics only

`text_raw` is preserved for embeddings.

In [ ]:
if not chunks_df.empty:
    if NLP is not None:
        lexical_texts = []
        for t in tqdm(chunks_df["text_raw"].fillna(""), desc="spaCy lexical preprocessing"):
            lexical_texts.append(build_text_lexical(t, max_chars=CONFIG["MAX_CHARS_FOR_SPACY"]))
        chunks_df["text_lexical"] = lexical_texts
    else:
        chunks_df["text_lexical"] = ""

## 9) Save base datasets + schema README

In [ ]:
# Keep pages CSV manageable by default
pages_to_save = pages_df.copy()
if not CONFIG["SAVE_FULL_PAGE_TEXT"] and "page_text_raw" in pages_to_save.columns:
    pages_to_save = pages_to_save.drop(columns=["page_text_raw"])

# Persist tables
documents_path = OUT_DIR / "documents.csv"
pages_path = OUT_DIR / "pages.csv"
chunks_path = OUT_DIR / "chunks.parquet"

for col in ["block_ids", "bbox", "suspected_header_footer_lines"]:
    if col in chunks_df.columns:
        chunks_df[col] = chunks_df[col].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x)
    if col in pages_to_save.columns:
        pages_to_save[col] = pages_to_save[col].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x)

documents_df.to_csv(documents_path, index=False)
pages_to_save.to_csv(pages_path, index=False)
chunks_df.to_parquet(chunks_path, index=False)

with open(OUT_DIR / "header_footer.json", "w", encoding="utf-8") as f:
    json.dump(header_footer_map, f, ensure_ascii=False, indent=2)

readme_text = f"""# Dutch Policy PDF Preprocessing Outputs

## Files
- documents.csv: one row per PDF (`doc_id`, filename, counts, extraction flags).
- pages.csv: one row per page with counts, cleaned page text without repeated header/footer lines, and suspected repeated lines.
- chunks.parquet: layout-aware chunks with traceability metadata, `text_raw`, and `text_lexical`.
- header_footer.json: per-document repeated line candidates used as header/footer heuristic.
- embeddings.npy: chunk embeddings aligned to chunks row order (if embeddings were computed).
- reports/cluster_report.md: cluster interpretability output (if clustering ran).

## Reproducibility
- Configure paths/thresholds/models in `CONFIG` at top of script.
- Run notebook top-to-bottom.
- Random seed: {CONFIG['RANDOM_SEED']}.
"""

with open(OUT_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

## 10) EDA + QA summary and plots (must-have)

In [ ]:
def save_hist(series: pd.Series, title: str, xlabel: str, outpath: Path, bins: int = 30) -> None:
    """Save histogram plot using matplotlib."""
    plt.figure(figsize=(8, 5))
    vals = series.dropna().values
    if len(vals) > 0:
        plt.hist(vals, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(outpath, dpi=140)
    plt.close()


n_docs = len(documents_df)
n_pages = len(pages_df)
n_chunks = len(chunks_df)
frac_failed_docs = float((~documents_df["extraction_ok"]).mean()) if n_docs else 0.0
frac_short_chunks = float((chunks_df["text_raw"].fillna("").str.len() < 30).mean()) if n_chunks else 0.0
frac_long_chunks = float((chunks_df["text_raw"].fillna("").str.len() > 1200).mean()) if n_chunks else 0.0

qa_summary = {
    "n_pdfs": n_docs,
    "n_pages": n_pages,
    "n_chunks": n_chunks,
    "frac_failed_docs": frac_failed_docs,
    "frac_short_chunks_lt30": frac_short_chunks,
    "frac_long_chunks_gt1200": frac_long_chunks,
}

with open(OUT_DIR / "reports" / "qa_summary.json", "w", encoding="utf-8") as f:
    json.dump(qa_summary, f, indent=2)

save_hist(documents_df["char_count_total"], "Document char count distribution", "chars per document", OUT_DIR / "plots" / "doc_char_hist.png")
save_hist(pages_df["page_char_count"], "Page char count distribution", "chars per page", OUT_DIR / "plots" / "page_char_hist.png")
if n_chunks:
    save_hist(chunks_df["text_raw"].fillna("").str.len(), "Chunk char count distribution", "chars per chunk", OUT_DIR / "plots" / "chunk_char_hist.png")

# Optional ydata-profiling report on metadata only
if YDATA_AVAILABLE and n_chunks:
    profile_cols = [c for c in ["doc_id", "page_start", "chunk_type_layout", "order_index", "char_start", "char_end"] if c in chunks_df.columns]
    if profile_cols:
        profile_df = chunks_df[profile_cols].copy()
        profile = ProfileReport(profile_df, title="Chunks Metadata Profile", minimal=True)
        profile.to_file(OUT_DIR / "reports" / "chunks_metadata_profile.html")
else:
    if not YDATA_AVAILABLE:
        warnings.warn("ydata-profiling not installed; skipping profile report.")

## 11) Embeddings + clustering (taxonomy bootstrapping)

In [ ]:
if SENTENCE_TRANSFORMERS_AVAILABLE and not chunks_df.empty:
    embed_model = SentenceTransformer(CONFIG["EMBED_MODEL"])
    texts = chunks_df["text_raw"].fillna("").tolist()
    embeddings = embed_model.encode(
        texts,
        batch_size=CONFIG["EMBED_BATCH_SIZE"],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    np.save(OUT_DIR / "embeddings.npy", embeddings)

    # KMeans labels
    for k in CONFIG["K_VALUES"]:
        if k <= 1 or k > len(chunks_df):
            warnings.warn(f"Skipping k={k}; invalid for n_chunks={len(chunks_df)}")
            continue
        km = KMeans(n_clusters=k, random_state=CONFIG["RANDOM_SEED"], n_init="auto")
        labels = km.fit_predict(embeddings)
        chunks_df[f"cluster_k{k}"] = labels

    # Optional HDBSCAN
    if HDBSCAN_AVAILABLE:
        try:
            hdb = hdbscan.HDBSCAN(min_cluster_size=CONFIG["HDBSCAN_MIN_CLUSTER_SIZE"])
            hlabels = hdb.fit_predict(embeddings)
            chunks_df["hdbscan_cluster"] = hlabels
            chunks_df["hdbscan_is_outlier"] = (hlabels == -1).astype(int)
        except Exception as e:
            warnings.warn(f"HDBSCAN failed: {repr(e)}")

    # PCA scatter for one selected k
    k_scatter = CONFIG["PCA_SCATTER_K"]
    k_col = f"cluster_k{k_scatter}"
    if k_col in chunks_df.columns:
        pca = PCA(n_components=2, random_state=CONFIG["RANDOM_SEED"])
        emb_2d = pca.fit_transform(embeddings)

        plt.figure(figsize=(8, 6))
        labels = chunks_df[k_col].values
        plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=labels, s=8, alpha=0.7)
        plt.title(f"PCA scatter of chunk embeddings ({k_col})")
        plt.xlabel("PC1")
        plt.ylabel("PC2")
        plt.tight_layout()
        plt.savefig(OUT_DIR / "plots" / f"pca_scatter_{k_col}.png", dpi=150)
        plt.close()

    # Save chunks with cluster labels
    chunks_df.to_parquet(chunks_path, index=False)
else:
    warnings.warn("Embeddings/clustering skipped (missing library or empty chunks).")

In [ ]:
# --- Auto-download outputs when cell finishes (Colab) ---

try:
    from google.colab import files
    import shutil
    import os

    zip_name = f"{OUT_DIR.name}.zip"
    zip_path = OUT_DIR.parent / zip_name

    # Remove existing zip if rerunning
    if zip_path.exists():
        zip_path.unlink()

    print(f"Zipping outputs to {zip_path} ...")
    shutil.make_archive(
        base_name=str(zip_path).replace(".zip", ""),
        format="zip",
        root_dir=str(OUT_DIR),
    )

    print("Starting download...")
    files.download(str(zip_path))

except ImportError:
    print("Not running in Google Colab — skipping auto-download.")

## 12) Cluster interpretability report

For each cluster:
- representative chunks (closest to centroid)
- top TF-IDF keywords on `text_lexical`

In [ ]:
def top_keywords_for_cluster(texts: List[str], top_n: int = 12) -> List[str]:
    """Get top TF-IDF terms for a cluster from lexical text."""
    clean_texts = [t for t in texts if isinstance(t, str) and t.strip()]
    if not clean_texts:
        return []
    vect = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
    X = vect.fit_transform(clean_texts)
    scores = np.asarray(X.mean(axis=0)).ravel()
    terms = np.array(vect.get_feature_names_out())
    top_idx = np.argsort(scores)[::-1][:top_n]
    return terms[top_idx].tolist()


def representative_chunk_indices(emb: np.ndarray, labels: np.ndarray, cluster_id: int, n_rep: int = 5) -> List[int]:
    """Indices of chunks closest to cluster centroid."""
    idx = np.where(labels == cluster_id)[0]
    if len(idx) == 0:
        return []
    cluster_emb = emb[idx]
    centroid = cluster_emb.mean(axis=0, keepdims=True)
    dists = ((cluster_emb - centroid) ** 2).sum(axis=1)
    order = np.argsort(dists)[:n_rep]
    return idx[order].tolist()


cluster_report_path = OUT_DIR / "reports" / "cluster_report.md"
cluster_summary_rows: List[Dict[str, Any]] = []

if (OUT_DIR / "embeddings.npy").exists() and any(c.startswith("cluster_k") for c in chunks_df.columns):
    emb = np.load(OUT_DIR / "embeddings.npy")
    cluster_cols = [c for c in chunks_df.columns if c.startswith("cluster_k")]
    report_lines = ["# Cluster Interpretability Report", ""]

    for ccol in cluster_cols:
        labels = chunks_df[ccol].values
        report_lines.append(f"## {ccol}")
        report_lines.append("")
        for cid in sorted(pd.Series(labels).dropna().unique().tolist()):
            cid = int(cid)
            mask = labels == cid
            cluster_size = int(mask.sum())
            texts_lex = chunks_df.loc[mask, "text_lexical"].fillna("").tolist()
            keywords = top_keywords_for_cluster(texts_lex, top_n=12)

            reps = representative_chunk_indices(emb, labels, cid, n_rep=5)
            report_lines.append(f"### Cluster {cid} (size={cluster_size})")
            report_lines.append(f"- Top keywords: {', '.join(keywords) if keywords else '(none)'}")
            report_lines.append("- Representative chunks:")

            for ridx in reps:
                row = chunks_df.iloc[ridx]
                snippet = row["text_raw"][:300].replace("\n", " ")
                report_lines.append(f"  - [{row['chunk_id']}] {snippet}")

            report_lines.append("")

            cluster_summary_rows.append(
                {
                    "cluster_column": ccol,
                    "cluster_id": cid,
                    "cluster_size": cluster_size,
                    "top_keywords": ", ".join(keywords),
                }
            )

    with open(cluster_report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))

    pd.DataFrame(cluster_summary_rows).to_csv(OUT_DIR / "reports" / "cluster_summary.csv", index=False)

## 13) Optional word clouds (TF-IDF weighted, per cluster)

Word clouds are **diagnostic visuals** for human sanity checks and cluster labeling.
They are not accuracy metrics.

In [ ]:
if WORDCLOUD_AVAILABLE and any(c.startswith("cluster_k") for c in chunks_df.columns):
    cluster_col = f"cluster_k{CONFIG['PCA_SCATTER_K']}"
    if cluster_col in chunks_df.columns:
        size_series = chunks_df[cluster_col].value_counts().sort_values(ascending=False)
        target_clusters = size_series.head(CONFIG["WORDCLOUD_TOP_CLUSTERS"]).index.tolist()

        for cid in target_clusters:
            texts = chunks_df.loc[chunks_df[cluster_col] == cid, "text_lexical"].fillna("").tolist()
            texts = [t for t in texts if t.strip()]
            if not texts:
                continue

            vec = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
            X = vec.fit_transform(texts)
            scores = np.asarray(X.mean(axis=0)).ravel()
            terms = np.array(vec.get_feature_names_out())
            freqs = {terms[i]: float(scores[i]) for i in np.argsort(scores)[::-1][:CONFIG["WORDCLOUD_MAX_WORDS"]]}

            wc = WordCloud(width=1400, height=800, background_color="white")
            wc.generate_from_frequencies(freqs)
            out_file = OUT_DIR / "wordclouds" / f"wordcloud_{cluster_col}_cluster{cid}.png"
            wc.to_file(str(out_file))
else:
    if not WORDCLOUD_AVAILABLE:
        warnings.warn("wordcloud not installed; skipping word cloud generation.")

## 14) Final save/preview cell

Prints output paths and previews core tables.

In [ ]:
# Re-save chunks after all optional enrichments
chunks_df.to_parquet(chunks_path, index=False)
chunks_df.head(10).to_csv(OUT_DIR / "chunks_preview.csv", index=False)

print("\n=== Pipeline complete ===")
print(f"Output directory: {OUT_DIR.resolve()}")
print(f"- {documents_path.name}")
print(f"- {pages_path.name}")
print(f"- {chunks_path.name}")
print("- header_footer.json")
print("- reports/*")
print("- plots/*")
if (OUT_DIR / "embeddings.npy").exists():
    print("- embeddings.npy")

print("\nDocuments preview:")
print(documents_df.head(5))
print("\nPages preview:")
print(pages_to_save.head(5))
print("\nChunks preview:")
print(chunks_df.head(5))

### RAG model with ColBERT

Code is based on the following tutorial:
[Tutorial](https://lightonai.github.io/pylate/#reranking)

In [ ]:
#!pip install pylate
#!pip install PyPDF2

from pylate import rank, models
from PyPDF2 import PdfReader
import os

In [ ]:
# I put the files in the current folder. This can be adapted to wherever you store a file
current_folder = os.getcwd()

def chunk_text(text: str, chunk_size: int = 128) -> list:
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

pdf_files = [f for f in os.listdir(current_folder) if f.endswith('.pdf')]


documents = []
chunk_ids = []
original_doc_ids = []
chunk_id = 0

for doc_id, pdf_file in enumerate(pdf_files):
    pdf_path = os.path.join(current_folder, pdf_file)
    with open(pdf_path, 'rb') as file:
        reader = PdfReader(file)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
        chunks = chunk_text(text)
        for chunk in chunks:
            documents.append(chunk)
            chunk_ids.append(chunk_id)
            original_doc_ids.append(doc_id)
            chunk_id += 1


queries = [
    "verplicht verplichtingen"]

model = models.ColBERT(
    model_name_or_path="LiquidAI/LFM2-ColBERT-350M",
)

queries_embeddings = model.encode(
    queries,
    is_query=True,
)

documents_embeddings = model.encode(
    documents,
    is_query=False,
)

reranked_chunks = rank.rerank(
    documents_ids=[chunk_ids],
    queries_embeddings=queries_embeddings,
    documents_embeddings=[documents_embeddings]
)


for query_idx, query in enumerate(queries):
    print(f"\nResults for query: '{query}'")
    for hit in reranked_chunks[query_idx]:
        chunk_id = hit['id']
        score = hit['score']
        original_doc_id = original_doc_ids[chunk_id]
        print(f"Original Document ID: {original_doc_id}, Chunk ID: {chunk_id}, Relevance Score: {score:.4f}")
        print(f"Content: {documents[chunk_id]}...\n")